Install Java

In [ ]:
# Install Java 8 (Hadoop requires Java)
!apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
!java -version


Download and Install Hadoop

In [ ]:
# Download Hadoop 3.3.6
!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz
!mv hadoop-3.3.6 /usr/local/hadoop
print("Hadoop installed successfully")


Set Hadoop Environment Variables

In [ ]:
import os
os.environ['JAVA_HOME']      = '/usr/lib/jvm/java-8-openjdk-amd64'
os.environ['HADOOP_HOME']    = '/usr/local/hadoop'
os.environ['PATH']           = os.environ['PATH'] + ':/usr/local/hadoop/bin:/usr/local/hadoop/sbin'
os.environ['HADOOP_CONF_DIR']= '/usr/local/hadoop/etc/hadoop'

# Verify Hadoop installation
!hadoop version


Configure Hadoop for Local Standalone Mode

In [ ]:
# In local standalone mode, NO XML config changes are needed.
# Hadoop runs as a single Java process using local filesystem (not HDFS).
print("Local standalone mode requires no extra configuration.")
print("Hadoop will use local filesystem directly.")


Create Sample Weather Data File

In [ ]:
# Create sample_weather.txt
# Columns: STN WBAN YEARMODA TEMP DEWP WDSP

weather_data = """STN    WBAN   YEARMODA  TEMP   DEWP   WDSP
123456 99999  20200101  72.5   55.3   12.4
123456 99999  20200102  68.2   50.1   10.2
123456 99999  20200103  75.8   58.6   15.3
123456 99999  20200104  80.1   62.4   18.7
123456 99999  20200105  65.3   48.9   8.5
123456 99999  20200106  70.4   53.2   11.6
123456 99999  20200107  78.9   60.5   14.8
123456 99999  20200108  82.3   65.1   20.2
123456 99999  20200109  69.7   52.8   9.9
123456 99999  20200110  74.6   57.4   13.5
123456 99999  20200111  77.2   59.8   16.1
123456 99999  20200112  63.5   47.2   7.8
123456 99999  20200113  71.8   54.6   12.0
123456 99999  20200114  85.4   67.3   22.4
123456 99999  20200115  66.9   51.0   10.7
"""

!mkdir -p /content/weather_input
with open('/content/weather_input/sample_weather.txt', 'w') as f:
    f.write(weather_data)

print("sample_weather.txt created successfully!")
print("\nFirst 5 lines of the weather file:")
!head -5 /content/weather_input/sample_weather.txt


Write WeatherMapper.java

In [ ]:
mapper_code = '''package com.weather;

import java.io.IOException;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.io.DoubleWritable;
import org.apache.hadoop.io.LongWritable;
import org.apache.hadoop.mapred.MapReduceBase;
import org.apache.hadoop.mapred.Mapper;
import org.apache.hadoop.mapred.OutputCollector;
import org.apache.hadoop.mapred.Reporter;

public class WeatherMapper extends MapReduceBase
        implements Mapper<LongWritable, Text, Text, DoubleWritable> {

    public void map(LongWritable key, Text value,
            OutputCollector<Text, DoubleWritable> output,
            Reporter reporter) throws IOException {
        String line = value.toString().trim();
        // Skip header line and empty lines
        if (line.startsWith("STN") || line.isEmpty()) return;
        String[] parts = line.split("\\\\s+");
        // Format: STN WBAN YEARMODA TEMP DEWP WDSP
        // Index:   0    1     2      3    4    5
        if (parts.length >= 6) {
            try {
                double temp = Double.parseDouble(parts[3]);
                double dewp = Double.parseDouble(parts[4]);
                double wdsp = Double.parseDouble(parts[5]);
                // Emit (field_name, value) pairs
                output.collect(new Text("TEMP"), new DoubleWritable(temp));
                output.collect(new Text("DEWP"), new DoubleWritable(dewp));
                output.collect(new Text("WDSP"), new DoubleWritable(wdsp));
            } catch (NumberFormatException e) {}
        }
    }
}
'''

!mkdir -p /content/weather
with open('/content/weather/WeatherMapper.java', 'w') as f:
    f.write(mapper_code)
print("WeatherMapper.java written.")


Write WeatherReducer.java

In [ ]:
reducer_code = '''package com.weather;

import java.io.IOException;
import java.util.Iterator;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.io.DoubleWritable;
import org.apache.hadoop.mapred.MapReduceBase;
import org.apache.hadoop.mapred.OutputCollector;
import org.apache.hadoop.mapred.Reducer;
import org.apache.hadoop.mapred.Reporter;

public class WeatherReducer extends MapReduceBase
        implements Reducer<Text, DoubleWritable, Text, Text> {

    public void reduce(Text key, Iterator<DoubleWritable> values,
            OutputCollector<Text, Text> output,
            Reporter reporter) throws IOException {
        double sum   = 0.0;
        int    count = 0;
        // Sum all values for this key (TEMP / DEWP / WDSP)
        while (values.hasNext()) {
            sum += values.next().get();
            count++;
        }
        double average = sum / count;
        // Emit (field_name, "Average = XX.XX")
        output.collect(key, new Text(String.format("Average = %.2f", average)));
    }
}
'''

with open('/content/weather/WeatherReducer.java', 'w') as f:
    f.write(reducer_code)
print("WeatherReducer.java written.")


Write WeatherRunner.java

In [ ]:
runner_code = '''package com.weather;

import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.io.DoubleWritable;
import org.apache.hadoop.mapred.FileInputFormat;
import org.apache.hadoop.mapred.FileOutputFormat;
import org.apache.hadoop.mapred.JobClient;
import org.apache.hadoop.mapred.JobConf;
import org.apache.hadoop.mapred.TextInputFormat;
import org.apache.hadoop.mapred.TextOutputFormat;

public class WeatherRunner {
    public static void main(String[] args) throws Exception {
        JobConf conf = new JobConf(WeatherRunner.class);
        conf.setJobName("WeatherAnalysis");

        conf.setMapperClass(WeatherMapper.class);
        conf.setReducerClass(WeatherReducer.class);

        // Map output types
        conf.setMapOutputKeyClass(Text.class);
        conf.setMapOutputValueClass(DoubleWritable.class);

        // Final output types
        conf.setOutputKeyClass(Text.class);
        conf.setOutputValueClass(Text.class);

        conf.setInputFormat(TextInputFormat.class);
        conf.setOutputFormat(TextOutputFormat.class);

        FileInputFormat.setInputPaths(conf, new Path(args[0]));
        FileOutputFormat.setOutputPath(conf, new Path(args[1]));

        JobClient.runJob(conf);
    }
}
'''

with open('/content/weather/WeatherRunner.java', 'w') as f:
    f.write(runner_code)
print("WeatherRunner.java written.")


Compile All Java Files

In [ ]:
!javac -classpath $(hadoop classpath) \
       -source 8 -target 8 \
       -d /content/weather \
       /content/weather/WeatherMapper.java \
       /content/weather/WeatherReducer.java \
       /content/weather/WeatherRunner.java

print("Compilation done.")


Create JAR File

In [ ]:
import os
os.chdir('/content/weather')
!jar -cvf weather.jar -C /content/weather .
print("JAR created.")


Run the MapReduce Job

In [ ]:
# Remove output dir if it exists from a previous run
!rm -rf /content/weather_output

!hadoop jar /content/weather/weather.jar \
            com.weather.WeatherRunner \
            /content/weather_input \
            /content/weather_output 2>&1

print("MapReduce job completed.")


View Output

In [ ]:
print("===== WEATHER ANALYSIS OUTPUT =====")
!cat /content/weather_output/part-*


In [ ]:
# ============================================================
# IMPORTANT NOTES FOR ORAL EXAM
# WEATHER DATA ANALYSIS USING MapReduce (Google Colab)
# ============================================================

# ------------------------------------------------------------
# WHAT IS THIS PRACTICAL ABOUT?
# ------------------------------------------------------------
# This practical computes the AVERAGE Temperature (TEMP),
# Dew Point (DEWP), and Wind Speed (WDSP) from a weather
# dataset using Hadoop MapReduce on Google Colab.
# It runs in LOCAL STANDALONE MODE - no HDFS, no cluster needed.
#
# Three Java files used:
#   WeatherMapper.java  - The Mapper class
#   WeatherReducer.java - The Reducer class
#   WeatherRunner.java  - The Driver/Runner class

# ------------------------------------------------------------
# WHAT IS THE INPUT FORMAT?
# ------------------------------------------------------------
# Space-separated weather file with columns:
#   STN    WBAN   YEARMODA  TEMP   DEWP   WDSP
#   123456 99999  20200101  72.5   55.3   12.4
# The header line (starts with STN) is skipped by the Mapper.

# ------------------------------------------------------------
# CODE EXPLANATION - WeatherMapper
# ------------------------------------------------------------
# implements Mapper<LongWritable, Text, Text, DoubleWritable>
#   - Input Key   : LongWritable - byte offset of the line
#   - Input Value : Text         - one full line of weather data
#   - Output Key  : Text         - field name (TEMP / DEWP / WDSP)
#   - Output Value: DoubleWritable - the numeric value
#
# map() method step by step:
#   line.startsWith("STN") -> skip the header row
#   line.split("\\s+")     -> split by whitespace into parts[]
#   parts[3] = TEMP, parts[4] = DEWP, parts[5] = WDSP
#   output.collect(new Text("TEMP"), new DoubleWritable(temp))
#     -> emits ("TEMP", 72.5), ("DEWP", 55.3), ("WDSP", 12.4) etc.

# ------------------------------------------------------------
# CODE EXPLANATION - WeatherReducer
# ------------------------------------------------------------
# implements Reducer<Text, DoubleWritable, Text, Text>
#   - Receives: ("TEMP", [72.5, 68.2, 75.8, ...])
#   - Sums all values and divides by count -> average
#   - Emits: ("TEMP", "Average = 73.51")
#
# reduce() method step by step:
#   while (values.hasNext()) { sum += values.next().get(); count++; }
#     -> accumulate sum and count
#   double average = sum / count;
#   output.collect(key, new Text(String.format("Average = %.2f", average)));
#     -> emit formatted average

# ------------------------------------------------------------
# CODE EXPLANATION - WeatherRunner
# ------------------------------------------------------------
# conf.setMapOutputKeyClass(Text.class);
# conf.setMapOutputValueClass(DoubleWritable.class);
#   -> Map emits (Text, DoubleWritable) - different from final output
#
# conf.setOutputKeyClass(Text.class);
# conf.setOutputValueClass(Text.class);
#   -> Final output is (Text, Text) - field name and average string

# ------------------------------------------------------------
# FULL MapReduce FLOW FOR WEATHER ANALYSIS
# ------------------------------------------------------------
# INPUT LINE:
#   123456 99999 20200101 72.5 55.3 12.4
#
# AFTER MAP (one line emits 3 pairs):
#   (TEMP, 72.5), (DEWP, 55.3), (WDSP, 12.4)
#   (TEMP, 68.2), (DEWP, 50.1), (WDSP, 10.2)  ... for each line
#
# AFTER SHUFFLE & SORT (automatic):
#   (DEWP, [55.3, 50.1, 58.6, ...])
#   (TEMP, [72.5, 68.2, 75.8, ...])
#   (WDSP, [12.4, 10.2, 15.3, ...])
#
# AFTER REDUCE:
#   DEWP    Average = 56.28
#   TEMP    Average = 73.51
#   WDSP    Average = 13.61

# ------------------------------------------------------------
# KEY CLASSES USED
# ------------------------------------------------------------
# LongWritable   : Hadoop serializable long. Used for line byte offset.
# DoubleWritable : Hadoop serializable double. Used for weather values.
# Text           : Hadoop serializable String. Used for field names.
# MapReduceBase  : Base class providing default implementations.
# Mapper         : Interface defining the map() method signature.
# Reducer        : Interface defining the reduce() method signature.
# OutputCollector: Collects (key, value) output from Mapper/Reducer.
# Reporter       : Reports job progress to Hadoop framework.
# JobConf        : Holds all configuration for a MapReduce job.
# JobClient      : Submits the job to Hadoop for execution.
#
# WHY DoubleWritable instead of double?
#   Hadoop needs to SERIALIZE data to send it across the network.
#   DoubleWritable implements the Writable interface for serialization.
#   Regular Java double cannot be directly serialized by Hadoop.

# ------------------------------------------------------------
# WHY -source 8 -target 8 IN COMPILATION?
# ------------------------------------------------------------
# Google Colab has Java 17 installed by default.
# Hadoop 3.3.6 on Colab runs using Java 8 (class file version 52.0).
# If compiled with Java 17, class file version = 61.0 -> ERROR.
# -source 8 -target 8 forces javac to produce Java 8 bytecode.

# ------------------------------------------------------------
# POSSIBLE ORAL EXAM QUESTIONS AND ANSWERS
# ------------------------------------------------------------
# Q: What is the purpose of this MapReduce program?
# A: To compute the average Temperature, Dew Point, and Wind Speed
#    from a weather dataset using Hadoop MapReduce.

# Q: What does the Mapper emit?
# A: For each data line it emits three key-value pairs:
#    ("TEMP", temp_value), ("DEWP", dewp_value), ("WDSP", wdsp_value)
#    using DoubleWritable for the values.

# Q: What does the Reducer do?
# A: Receives all values for a key (e.g., all TEMP readings),
#    sums them, divides by count, and emits the average.

# Q: Why are map output types different from final output types?
# A: The Mapper emits (Text, DoubleWritable) for numeric computation.
#    The Reducer outputs (Text, Text) because the final result is a
#    formatted string like "Average = 73.51".
#    Both must be declared separately in the Runner using
#    setMapOutputKeyClass/setMapOutputValueClass.

# Q: What is Shuffle and Sort?
# A: The automatic phase between Map and Reduce.
#    All intermediate (key, value) pairs are sorted by key and
#    all values for the same key are grouped and sent to the same Reducer.

# Q: What is the expected output?
# A: DEWP    Average = 56.28
#    TEMP    Average = 73.51
#    WDSP    Average = 13.61
